# Week 3 - Original BSM Chooser Option Model Replication

## Objective

Implement and validate a Black-Scholes-Merton (BSM) simple chooser option model using:

1. the paper's published JPM parameter set;
2. the processed Week 2 JPM market dataset; and
3. transparent analytical and Monte Carlo validation.

The required contract settings are **strike price $150**, **choice time 0.5 year**, and **final maturity 1 year**.

### References

- Huang, Wang, and Wan (2021), *Exploration of JPMorgan Chooser Option Pricing*, Table 2 and Table 3, DOI: 10.54691/bcpbm.v15i.224.
- Hull (2021), *Options, Futures, and Other Derivatives*, 11th ed., Section 26.8 (simple chooser option decomposition).


## 1. Required Packages

In [1]:
import json
from pathlib import Path
from math import erf, exp, log, sqrt

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

CONFIG_PATH = Path("./parameter_config.json")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Parameter configuration not found: {CONFIG_PATH.resolve()}")

with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
    CONFIG = json.load(config_file)

print(f"Loaded parameter configuration from: {CONFIG_PATH.resolve()}")


Loaded parameter configuration from: G:\JPM-Chooser Option Pricing\Week 3\parameter_config.json


## 2. Link to the Week 2 Processed Dataset

The original Week 3 notebook used manually selected values and did not connect to Week 2. This section loads the processed dataset and extracts the latest valid JPM close, Treasury rate, and 20-day annualized historical volatility.


In [2]:
configured_data_path = Path(CONFIG["week2_updated_baseline"]["data_file"])
DATA_CANDIDATES = [
    configured_data_path,
    Path("Week2/processed_data/market_data_processed.csv"),
]

try:
    WEEK2_DATA_PATH = next(path for path in DATA_CANDIDATES if path.exists())
except StopIteration as exc:
    raise FileNotFoundError(
        "Could not locate Week 2 processed data. Run this notebook from the Week 3 folder "
        "or update DATA_CANDIDATES."
    ) from exc

market_data = pd.read_csv(WEEK2_DATA_PATH, parse_dates=["Date"])

required_columns = {
    "Date",
    "Close",
    "Treasury_Rate",
    "Rolling_Volatility_20D",
}
missing_columns = required_columns.difference(market_data.columns)
if missing_columns:
    raise KeyError(f"Week 2 dataset is missing columns: {sorted(missing_columns)}")

latest_market_row = (
    market_data
    .dropna(subset=["Close", "Treasury_Rate", "Rolling_Volatility_20D"])
    .sort_values("Date")
    .iloc[-1]
)

week2_linkage = pd.DataFrame(
    {
        "Field": ["Date", "JPM close", "Treasury rate (%)", "20-day volatility"],
        "Value": [
            latest_market_row["Date"].date().isoformat(),
            latest_market_row["Close"],
            latest_market_row["Treasury_Rate"],
            latest_market_row["Rolling_Volatility_20D"],
        ],
        "Week 2 Column": [
            "Date",
            "Close",
            "Treasury_Rate",
            "Rolling_Volatility_20D",
        ],
    }
)

print(f"Loaded {len(market_data):,} rows from {WEEK2_DATA_PATH.resolve()}")
week2_linkage


Loaded 1,760 rows from G:\JPM-Chooser Option Pricing\Week2\processed_data\market_data_processed.csv


               Field       Value           Week 2 Column
0               Date  2024-12-30                    Date
1          JPM close  231.077560                   Close
2  Treasury rate (%)    4.550000           Treasury_Rate
3  20-day volatility    0.190625  Rolling_Volatility_20D

## 3. Parameter Configuration

Two configurations are kept separate:

- **Paper replication:** exactly matches the values reported in Table 2 of Huang et al. (2021).
- **Week 2 updated baseline:** uses the latest processed JPM close, Treasury rate, and rolling volatility. Week 2 did not collect dividend yield, so the paper's 2.33% yield is retained and explicitly labelled rather than silently set to zero.

Rates and volatility are stored as annual decimals in the model.


In [3]:
contract_config = CONFIG["contract"]
paper_config = CONFIG["paper_replication"]
week2_config = CONFIG["week2_updated_baseline"]

PAPER_PARAMS = {
    "S0": float(paper_config["S0"]),
    "K": float(contract_config["K"]),
    "r": float(paper_config["r"]),
    "q": float(paper_config["q"]),
    "sigma": float(paper_config["sigma"]),
    "T1": float(contract_config["T1"]),
    "T2": float(contract_config["T2"]),
}

WEEK2_PARAMS = {
    "S0": float(latest_market_row[week2_config["spot_column"]]),
    "K": float(contract_config["K"]),
    "r": float(latest_market_row[week2_config["rate_column"]]) * float(week2_config["rate_scale"]),
    "q": float(week2_config["dividend_yield"]),
    "sigma": float(latest_market_row[week2_config["volatility_column"]]),
    "T1": float(contract_config["T1"]),
    "T2": float(contract_config["T2"]),
}

parameter_configuration = pd.DataFrame(
    [
        {
            "Configuration": "Paper replication",
            **PAPER_PARAMS,
            "Market date": paper_config["market_date"],
            "Source": "Huang et al. (2021), Table 2",
        },
        {
            "Configuration": "Week 2 updated baseline",
            **WEEK2_PARAMS,
            "Market date": latest_market_row["Date"].date().isoformat(),
            "Source": "Week 2 processed data; q retained from paper",
        },
    ]
)

parameter_configuration


             Configuration  ...                                        Source
0        Paper replication  ...                  Huang et al. (2021), Table 2
1  Week 2 updated baseline  ...  Week 2 processed data; q retained from paper

[2 rows x 10 columns]

## 4. Reusable BSM Functions

In [4]:
def normal_cdf(x):
    # Standard normal cumulative distribution function.
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


def validate_model_inputs(S, K, sigma, T):
    if S <= 0:
        raise ValueError("Underlying price S must be positive.")
    if K <= 0:
        raise ValueError("Strike price K must be positive.")
    if sigma <= 0:
        raise ValueError("Volatility sigma must be positive.")
    if T <= 0:
        raise ValueError("Time to maturity T must be positive.")


def calculate_d1_d2(S, K, r, q, sigma, T):
    validate_model_inputs(S, K, sigma, T)
    d1 = (log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return d1, d2


def bsm_call_price(S, K, r, q, sigma, T):
    d1, d2 = calculate_d1_d2(S, K, r, q, sigma, T)
    return S * exp(-q * T) * normal_cdf(d1) - K * exp(-r * T) * normal_cdf(d2)


def bsm_put_price(S, K, r, q, sigma, T):
    d1, d2 = calculate_d1_d2(S, K, r, q, sigma, T)
    return K * exp(-r * T) * normal_cdf(-d2) - S * exp(-q * T) * normal_cdf(-d1)


## 5. Simple Chooser Option Model

At choice time `T1`, the holder selects the more valuable of a European call and put with common strike `K` and maturity `T2`.

Using put-call parity, the simple chooser is equivalent to:

\[
\text{Chooser}_0 = C(S_0,K,T_2) + e^{-q(T_2-T_1)}P(S_0,K^*,T_1),
\]

where

\[
K^* = K e^{-(r-q)(T_2-T_1)}.
\]

The corresponding choice boundary at `T1` is `K*`, not generally `K`. This distinction matters when interest rates or dividend yields are non-zero.


In [5]:
def chooser_choice_boundary(K, r, q, T1, T2):
    if not 0 < T1 < T2:
        raise ValueError("Choice time must satisfy 0 < T1 < T2.")
    return K * exp(-(r - q) * (T2 - T1))


def simple_chooser_price(S, K, r, q, sigma, T1, T2):
    validate_model_inputs(S, K, sigma, T2)
    critical_strike = chooser_choice_boundary(K, r, q, T1, T2)
    call_component = bsm_call_price(S, K, r, q, sigma, T2)
    put_component = exp(-q * (T2 - T1)) * bsm_put_price(
        S, critical_strike, r, q, sigma, T1
    )
    return call_component + put_component


def price_configuration(name, params):
    call = bsm_call_price(params['S0'], params['K'], params['r'], params['q'], params['sigma'], params['T2'])
    put = bsm_put_price(params["S0"], params["K"], params["r"], params["q"], params["sigma"], params["T2"])
    chooser = simple_chooser_price(
        params["S0"], params["K"], params["r"], params["q"],
        params["sigma"], params["T1"], params["T2"]
    )
    boundary = chooser_choice_boundary(params["K"], params["r"], params["q"], params["T1"], params["T2"])
    return {
        "Configuration": name,
        "European Call": call,
        "European Put": put,
        "Chooser Option": chooser,
        "Choice Boundary at T1": boundary,
    }


pricing_results = pd.DataFrame(
    [
        price_configuration("Paper replication", PAPER_PARAMS),
        price_configuration("Week 2 updated baseline", WEEK2_PARAMS),
    ]
)

pricing_results


             Configuration  ...  Choice Boundary at T1
0        Paper replication  ...             151.643943
1  Week 2 updated baseline  ...             148.344207

[2 rows x 5 columns]

## 6. Comparison with the Paper's Reported Results

The paper reports ten simulated paths in Table 3, but it does **not** report the random seed or the random shocks used to generate those paths. Therefore, a scientifically valid review can verify the reported payoff arithmetic exactly, but cannot claim row-for-row regeneration of the same stock-price paths.

The paper chooses `CALL` when the six-month price is above $150 and `PUT` otherwise. The following comparison reproduces its published table and recalculates each terminal payoff.


In [6]:
paper_reported = pd.DataFrame(
    {
        "First_6M_Stock": [118.33, 222.63, 186.53, 164.08, 159.09, 186.73, 106.61, 163.06, 129.26, 115.41],
        "Paper_Choice": ["PUT", "CALL", "CALL", "CALL", "CALL", "CALL", "PUT", "CALL", "PUT", "PUT"],
        "Second_6M_Stock": [116.77, 192.89, 192.94, 148.77, 116.78, 128.12, 90.52, 179.61, 144.82, 136.50],
        "Paper_Payoff": [33.23, 42.89, 42.94, 0.00, 0.00, 0.00, 59.48, 29.61, 5.18, 13.50],
    }
)

paper_reported["Recalculated_Choice"] = np.where(
    paper_reported["First_6M_Stock"] > PAPER_PARAMS["K"], "CALL", "PUT"
)
paper_reported["Recalculated_Payoff"] = np.where(
    paper_reported["Paper_Choice"].eq("CALL"),
    np.maximum(paper_reported["Second_6M_Stock"] - PAPER_PARAMS["K"], 0.0),
    np.maximum(PAPER_PARAMS["K"] - paper_reported["Second_6M_Stock"], 0.0),
)
paper_reported["Payoff_Absolute_Error"] = (
    paper_reported["Recalculated_Payoff"] - paper_reported["Paper_Payoff"]
).abs()

paper_reported


   First_6M_Stock Paper_Choice  ...  Recalculated_Payoff  Payoff_Absolute_Error
0      118.330000          PUT  ...            33.230000               0.000000
1      222.630000         CALL  ...            42.890000               0.000000
2      186.530000         CALL  ...            42.940000               0.000000
3      164.080000         CALL  ...             0.000000               0.000000
4      159.090000         CALL  ...             0.000000               0.000000
5      186.730000         CALL  ...             0.000000               0.000000
6      106.610000          PUT  ...            59.480000               0.000000
7      163.060000         CALL  ...            29.610000               0.000000
8      129.260000          PUT  ...             5.180000               0.000000
9      115.410000          PUT  ...            13.500000               0.000000

[10 rows x 7 columns]

## 7. Monte Carlo Validation Against the Closed-Form Price

The simulation uses the same paper parameters, two six-month periods, risk-neutral GBM dynamics, and a fixed seed for reproducibility. At `T1`, the theoretically correct boundary from put-call parity determines whether the terminal payoff is a call or a put.


In [7]:
def monte_carlo_chooser_price(params, n_paths=250_000, seed=20260806):
    rng = np.random.default_rng(seed)
    dt1 = params["T1"]
    dt2 = params["T2"] - params["T1"]
    z1 = rng.standard_normal(n_paths)
    z2 = rng.standard_normal(n_paths)

    drift1 = (params["r"] - params["q"] - 0.5 * params["sigma"]**2) * dt1
    drift2 = (params["r"] - params["q"] - 0.5 * params["sigma"]**2) * dt2
    diffusion1 = params["sigma"] * sqrt(dt1) * z1
    diffusion2 = params["sigma"] * sqrt(dt2) * z2

    stock_t1 = params["S0"] * np.exp(drift1 + diffusion1)
    stock_t2 = stock_t1 * np.exp(drift2 + diffusion2)

    boundary = chooser_choice_boundary(
        params["K"], params["r"], params["q"], params["T1"], params["T2"]
    )
    choose_call = stock_t1 >= boundary
    terminal_payoff = np.where(
        choose_call,
        np.maximum(stock_t2 - params["K"], 0.0),
        np.maximum(params["K"] - stock_t2, 0.0),
    )

    discount = exp(-params["r"] * params["T2"])
    price = discount * terminal_payoff.mean()
    standard_error = discount * terminal_payoff.std(ddof=1) / sqrt(n_paths)
    return {
        "Paths": n_paths,
        "Seed": seed,
        "Monte Carlo Price": price,
        "Standard Error": standard_error,
        "95% CI Lower": price - 1.96 * standard_error,
        "95% CI Upper": price + 1.96 * standard_error,
        "Call Choice Share": choose_call.mean(),
    }


mc_config = CONFIG["monte_carlo"]
mc_result = monte_carlo_chooser_price(
    PAPER_PARAMS,
    n_paths=int(mc_config["n_paths"]),
    seed=int(mc_config["seed"]),
)
closed_form_paper_price = simple_chooser_price(
    PAPER_PARAMS["S0"], PAPER_PARAMS["K"], PAPER_PARAMS["r"],
    PAPER_PARAMS["q"], PAPER_PARAMS["sigma"], PAPER_PARAMS["T1"], PAPER_PARAMS["T2"]
)

monte_carlo_validation = pd.DataFrame(
    [
        {
            **mc_result,
            "Closed-Form Price": closed_form_paper_price,
            "Absolute Difference": abs(mc_result["Monte Carlo Price"] - closed_form_paper_price),
            "Within 95% CI": mc_result["95% CI Lower"] <= closed_form_paper_price <= mc_result["95% CI Upper"],
        }
    ]
)

monte_carlo_validation


    Paths      Seed  ...  Absolute Difference  Within 95% CI
0  250000  20260806  ...             0.050431           True

[1 rows x 10 columns]

## 8. Model Validation Summary

In [8]:
paper_call = bsm_call_price(
    PAPER_PARAMS["S0"], PAPER_PARAMS["K"], PAPER_PARAMS["r"],
    PAPER_PARAMS["q"], PAPER_PARAMS["sigma"], PAPER_PARAMS["T2"]
)
paper_put = bsm_put_price(
    PAPER_PARAMS["S0"], PAPER_PARAMS["K"], PAPER_PARAMS["r"],
    PAPER_PARAMS["q"], PAPER_PARAMS["sigma"], PAPER_PARAMS["T2"]
)
paper_chooser = closed_form_paper_price

parity_error = abs(
    (paper_call - paper_put)
    - (
        PAPER_PARAMS["S0"] * exp(-PAPER_PARAMS["q"] * PAPER_PARAMS["T2"])
        - PAPER_PARAMS["K"] * exp(-PAPER_PARAMS["r"] * PAPER_PARAMS["T2"])
    )
)
max_paper_payoff_error = paper_reported["Payoff_Absolute_Error"].max()
paper_choice_matches = paper_reported["Paper_Choice"].eq(
    paper_reported["Recalculated_Choice"]
).all()
mc_difference = monte_carlo_validation.loc[0, "Absolute Difference"]
mc_standard_error = monte_carlo_validation.loc[0, "Standard Error"]

validation_summary = pd.DataFrame(
    {
        "Validation Test": [
            "Put-call parity",
            "Chooser >= call",
            "Chooser >= put",
            "Chooser <= call + put",
            "Paper Table 3 choices follow the paper rule",
            "Paper Table 3 payoffs recalculate exactly",
            "Monte Carlo is within three standard errors of closed form",
        ],
        "Observed": [
            parity_error,
            paper_chooser - paper_call,
            paper_chooser - paper_put,
            paper_call + paper_put - paper_chooser,
            int(paper_choice_matches),
            max_paper_payoff_error,
            mc_difference / mc_standard_error,
        ],
        "Pass": [
            parity_error < 1e-10,
            paper_chooser >= paper_call,
            paper_chooser >= paper_put,
            paper_chooser <= paper_call + paper_put,
            paper_choice_matches,
            max_paper_payoff_error < 1e-10,
            mc_difference <= 3.0 * mc_standard_error,
        ],
    }
)

validation_summary


                                     Validation Test  Observed  Pass
0                                    Put-call parity  0.000000  True
1                                    Chooser >= call 10.440927  True
2                                     Chooser >= put 13.756855  True
3                              Chooser <= call + put  4.932143  True
4        Paper Table 3 choices follow the paper rule  1.000000  True
5          Paper Table 3 payoffs recalculate exactly  0.000000  True
6  Monte Carlo is within three standard errors of...  0.822757  True

In [9]:
assert validation_summary["Pass"].all(), "At least one validation test failed."
print("All analytical, paper-comparison, and Monte Carlo validation tests passed.")


All analytical, paper-comparison, and Monte Carlo validation tests passed.


## 9. Save Deliverables

This cell creates the required parameter configuration file together with reproducible pricing and validation outputs.


In [10]:
OUTPUT_DIR = Path("./model_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

parameter_configuration.to_csv(OUTPUT_DIR / "parameter_configuration.csv", index=False)
pricing_results.to_csv(OUTPUT_DIR / "week3_pricing_results.csv", index=False)
paper_reported.to_csv(OUTPUT_DIR / "paper_table3_comparison.csv", index=False)
monte_carlo_validation.to_csv(OUTPUT_DIR / "monte_carlo_validation.csv", index=False)
validation_summary.to_csv(OUTPUT_DIR / "validation_summary.csv", index=False)

saved_files = pd.DataFrame(
    {
        "Deliverable": [
            "Parameter configuration",
            "BSM and chooser prices",
            "Paper result comparison",
            "Monte Carlo comparison",
            "Validation summary",
        ],
        "File": [
            "parameter_configuration.csv",
            "week3_pricing_results.csv",
            "paper_table3_comparison.csv",
            "monte_carlo_validation.csv",
            "validation_summary.csv",
        ],
    }
)

print(f"Saved results to: {OUTPUT_DIR.resolve()}")
saved_files


Saved results to: G:\JPM-Chooser Option Pricing\Week 3\model_results


               Deliverable                         File
0  Parameter configuration  parameter_configuration.csv
1   BSM and chooser prices    week3_pricing_results.csv
2  Paper result comparison  paper_table3_comparison.csv
3   Monte Carlo comparison   monte_carlo_validation.csv
4       Validation summary       validation_summary.csv

# Summary

The Week 3 requirements are now addressed with auditable evidence:

- The BSM call, put, and simple chooser option functions are reusable and validate their inputs.
- The paper parameter set uses `S0=156.70`, `K=150`, `r=0.15%`, `q=2.33%`, `sigma=28.2%`, `T1=0.5`, and `T2=1`.
- The Week 2 updated baseline is linked directly to the processed dataset instead of using invented market inputs.
- The paper's ten reported payoffs are checked row by row.
- A reproducible two-period Monte Carlo price is compared with the analytical chooser price.
- A standalone parameter configuration file and validation result files are exported.

### Important limitation

The paper does not disclose the random seed or random shocks behind Table 3. Its ten stock-price paths therefore cannot be regenerated exactly from the published information. The payoff comparison is exact; the Monte Carlo comparison is distributional and uses a documented fixed seed.
